# GPT-2 project tour

This orientation checks the new project environment and shows where experiments belong. It does not build or train GPT-2 yet. The first model lesson will bridge the differences from GPT-1.

Run `scripts/setup-gpt2-python.ps1` from the repository root first. Select **Karpathy GPT-2 (project .venv)** as the kernel. This notebook and the CLI import the same editable Python package.

See the [project guide](../../README.md), [learning plan](../../docs/learning-plan.md), and [experiment guide](../../experiments/README.md).


In [ ]:
from pathlib import Path
import sys

from gpt2_lab.config import ModelConfig, load_experiment

candidates = [Path.cwd(), Path.cwd() / "gpt-2", *Path.cwd().parents]
tutorial_root = next(
    (candidate for candidate in candidates
     if (candidate / "experiments/configs/tiny-playground.json").is_file()),
    None,
)
if tutorial_root is None:
    raise FileNotFoundError("Open the repository, gpt-2 folder, or its notebook folder and restart the kernel.")

print(f"Python: {sys.executable}")
print(f"Tutorial: {tutorial_root}")


## Inspect an experiment

The JSON describes a dataset, model dimensions, and intended training settings. Its dataset path is relative to the JSON file, so changing the launch folder does not select different data.

Loading settings allocates no model. The vocabulary of 256 here is a small dimension placeholder; no tokenizer has been implemented. The sample corpus contains fictional sentences for environment checks.


In [ ]:
import json

experiment = load_experiment(tutorial_root / "experiments/configs/tiny-playground.json")
print(json.dumps(experiment.plan(), indent=2))
assert experiment.dataset_path.is_file()


## Compare model dimensions across languages

`ModelConfig` describes shapes only. GPT-2's head size is the embedding width divided by the number of heads. These presets can be inspected without loading 124 million parameters.

C equivalents: [model_config.h](../../c/include/model_config.h), [model_config.c](../../c/src/model_config.c), and [main.c](../../c/src/main.c).

Java equivalents: [ModelConfig.java](../../java/src/main/java/learning/karpathy/gpt2/ModelConfig.java) and [Main.java](../../java/src/main/java/learning/karpathy/gpt2/Main.java).

Run either companion with the `tiny` or `gpt2` preset using the [run instructions](../../README.md#start-here). Their presets are introductory checks; they do not consume the experiment JSON yet.


In [ ]:
for name, config in (("tiny", ModelConfig()), ("gpt2", ModelConfig.gpt2_124m())):
    print(f"{name}: vocab={config.vocab_size} context={config.context_length} "
          f"layers={config.n_layer} heads={config.n_head} "
          f"embedding={config.n_embd} head_size={config.head_size}")


## Make a small variation

A frozen configuration is a baseline. Use `replace` to explore a different setting without mutating that baseline. A persistent experiment goes in a copied JSON file with its own name and description.

Try setting the width to 31: the configuration should reject it because four attention heads cannot divide that width equally. Then try a different number of heads that does divide the width.


In [ ]:
from dataclasses import replace

baseline = experiment.config.model
wider = replace(baseline, n_embd=64)
print(f"Baseline: width={baseline.n_embd}, head size={baseline.head_size}")
print(f"Variation: width={wider.n_embd}, head size={wider.head_size}")
assert baseline.n_embd == 32  # The original experiment settings remain unchanged.


## Record a future experiment

From the repository root, run:

```powershell
.\scripts\run-gpt2.ps1 -Language python -Action prepare-run
```

This creates a unique ignored folder under `gpt-2/runs/` containing `config.json` and `manifest.json`. It records settings, input hashes, Git revision/dirty flag, and Python runtime. Its status is **prepared**: no training has run.

The model and training lessons will add actual metrics, checkpoints, generated text, and resume support. Custom corpora belong under `data/local/`; see the [experiment guide](../../experiments/README.md). Dataset, tokenizer, architecture, and experiment settings will be connected as the relevant lessons arrive.
